# Benchmarks on an Azure ML node with a GPU

Runs three packages on the same circuits, on one node of an Azure ML cluster, and collects the
results:

| Part | What runs | On |
| --- | --- | --- |
| `julia` | PauliPropagation.jl 0.9.2, `tools/julia/benchmark.jl` | CPU, 1 thread and all cores |
| `paulipropagation_qdk` | this package from PyPI, `benchmarks/run.py` | CPU, 1 thread, 8 threads and all cores |
| `cupauliprop` | cuPauliProp of cuQuantum 26.11, `benchmarks/cupauliprop.py` | GPU |

Every part writes its timings, numbers of terms and overlaps to a JSON file. The parts are
independent: one that fails does not stop the others.

| Run | Cases | Node time |
| --- | --- | --- |
| Trial | the two small ones | about 15 minutes, most of it starting the node and installing Julia |
| Full | all four, and the large one of 131,664,350 terms | about 1.5 hours, limited to 2.5 hours |

The trial run shows whether the environment builds, the GPU is visible and the three parts
work, before the full run is paid for. A cluster of low-priority nodes is enough: the timings
do not depend on the priority, and a run that is pre-empted is submitted again.

## Setup

```bash
pip install -r requirements.txt
az login
cp .env.example .env    # then fill in the workspace and the name of the cluster
```

In [ ]:
import json
import os
import shutil
import tempfile
import time
from pathlib import Path

from azure.ai.ml import MLClient, Output, command
from azure.ai.ml.entities import Environment
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()
cluster_name = os.environ["CLUSTER_NAME"]
price_per_hour = float(os.getenv("PRICE_PER_HOUR", "2"))

HERE = Path.cwd()
ROOT = next(folder for folder in (HERE, *HERE.parents) if (folder / "benchmarks" / "run.py").exists())

SMALL = ["readme_tfi_32", "kicked_ising_eagle_127"]
ALL = [*SMALL, "tilted_ising_6x6", "noisy_ising_6x6", "noisy_ising_6x6_large"]

## Workspace and cluster

The cluster has to exist. For the cost, check that it scales down to zero nodes and how long
an idle node is kept.

In [ ]:
ml_client = MLClient(
    DefaultAzureCredential(),
    os.environ["SUBSCRIPTION_ID"],
    os.environ["RESOURCE_GROUP"],
    os.environ["WORKSPACE_NAME"],
)
cluster = ml_client.compute.get(cluster_name)
print(f"{cluster.name}: {cluster.size}, {cluster.tier}")
print(f"{cluster.min_instances} to {cluster.max_instances} nodes, an idle node is kept for "
      f"{cluster.idle_time_before_scale_down} s")

## The job

Only the files that the job needs are uploaded: the cases, the three benchmark scripts and
the script that runs them on the node.

In [ ]:
FILES = [
    "benchmarks/run.py",
    "benchmarks/cupauliprop.py",
    "benchmarks/azureml/job.sh",
    "tools/julia/benchmark.jl",
    "tools/julia/circuitjson.jl",
    *(f"benchmarks/cases/{name}.json" for name in ALL),
]

environment = Environment(
    image="mcr.microsoft.com/azureml/openmpi5.0-cuda12.4-ubuntu22.04:latest",
    conda_file=ROOT / "benchmarks" / "azureml" / "environment.yml",
)


def staged() -> Path:
    """A temporary folder with the files of the job, laid out as in the repository."""
    folder = Path(tempfile.mkdtemp(prefix="paulipropagation-qdk-"))
    for file in FILES:
        (folder / file).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(ROOT / file, folder / file)
    return folder


def submit(label: str, cases: list[str], minutes: int):
    """Submits the job for `cases`, which Azure ML ends after `minutes`."""
    job = command(
        code=staged(),
        command="bash benchmarks/azureml/job.sh ${{outputs.results}} " + " ".join(cases),
        outputs={"results": Output(type="uri_folder")},
        environment=environment,
        compute=cluster_name,
        timeout=60 * minutes,
        experiment_name="paulipropagation-qdk-benchmarks",
        display_name=f"{label}: {len(cases)} cases",
    )
    job = ml_client.jobs.create_or_update(job)
    print(f"{job.name}: {job.studio_url}")
    return job


def wait(job) -> float:
    """Shows the output of the job until it ends. Returns the seconds this took."""
    start = time.time()
    try:
        ml_client.jobs.stream(job.name)
    except Exception as error:  # a part of the job failed: its other results are still there
        print(error)
    return time.time() - start

## The results

`collect` downloads the results of a job. `show` prints the timings, how far the three
results are apart, and what the node cost.

In [ ]:
def collect(job) -> Path:
    """Downloads the results of the job and returns their folder."""
    target = Path(tempfile.mkdtemp(prefix="paulipropagation-qdk-results-"))
    ml_client.jobs.download(job.name, download_path=target, output_name="results")
    return next(target.rglob("machine.log")).parent


def load(folder: Path, name: str) -> dict:
    file = folder / f"{name}.json"
    return json.loads(file.read_text()) if file.exists() else {}


def seconds(value: float | None) -> str:
    if value is None:
        return ""
    return f"{value:.2f} s" if value >= 1 else f"{1000 * value:.1f} ms"


def table(header: list[str], rows: list[list]) -> None:
    lines = [header, ["---"] + ["---:"] * (len(header) - 1), *rows]
    display(Markdown("\n".join("| " + " | ".join(map(str, line)) + " |" for line in lines)))


def show(folder: Path, waited: float | None = None) -> None:
    julia, ours, gpu, job = (load(folder, name) for name in ("julia", "paulipropagation_qdk", "cupauliprop", "job"))
    names = [name for name in ALL if any(name in report.get("cases", {}) for report in (julia, ours, gpu))]
    print((folder / "machine.log").read_text())

    rows, agreement = [], []
    for name in names:
        j, o, g = (report.get("cases", {}).get(name, {}) for report in (julia, ours, gpu))
        j_seconds, o_seconds, g_seconds = (case.get("seconds", {}) for case in (j, o, g))
        one = [s for label, s in j_seconds.items() if label == "dict" or "1 thread" in label]
        many = [s for label, s in j_seconds.items() if label != "dict" and "1 thread" not in label]
        most = max(o_seconds, key=int, default="")
        rows.append([
            name,
            f"{o.get('terms', j.get('terms', g.get('terms', 0))):,}",
            seconds(o_seconds.get("1")),
            seconds(min(one, default=None)),
            seconds(o_seconds.get("8")),
            seconds(o_seconds.get(most)) + (f" ({most})" if most else ""),
            seconds(min(many, default=None)),
            seconds(g_seconds.get("buffers per gate")),
            seconds(g_seconds.get("two buffers")),
        ])
        reference = o or g.get("this_package", {})
        difference = g.get("difference", {})
        agreement.append([
            name,
            f"{reference.get('terms', 0):,}",
            f"{j['terms'] - reference['terms']:+,}" if j and reference else "",
            f"{abs(j['overlap_with_zero'] - reference['overlap_with_zero']):.1e}" if j and reference else "",
            f"{g['terms'] - reference['terms']:+,}" if g and reference else "",
            f"{abs(g['overlap_with_zero'] - reference['overlap_with_zero']):.1e}" if g and reference else "",
            f"{abs(g['norm'] - g['this_package']['norm']):.1e}" if g else "",
            f"{difference['only_this_package']} / {difference['only_cupauliprop']}" if difference else "",
            f"{difference['largest_coeff_difference']:.1e}" if difference else "",
        ])

    display(Markdown(f"**Timings**, best of three. {julia.get('threads', '')} threads for Julia, "
                     f"{gpu.get('gpu', 'no GPU result')}, cuQuantum {gpu.get('version', '')}."))
    table(["Case", "Terms", "this package, 1 thread", "Julia, 1 thread", "this package, 8 threads",
           "this package, most threads", "Julia, all threads", "cuPauliProp, buffers per gate",
           "cuPauliProp, two buffers"], rows)
    display(Markdown("**Agreement** with this package: difference in the number of terms and in the "
                     "overlap with the zero state, and for cuPauliProp also in the norm, the terms "
                     "that only one of the two has, and the largest difference of a coefficient."))
    table(["Case", "Terms", "Julia, terms", "Julia, overlap", "cuPauliProp, terms", "cuPauliProp, overlap",
           "cuPauliProp, norm", "only this package / only cuPauliProp", "coefficient"], agreement)

    if job:
        print(f"failed parts: {', '.join(job['failed']) or 'none'}")
        cost = f"the script ran for {job['seconds'] / 60:.0f} min, {job['seconds'] / 3600 * price_per_hour:.2f} at {price_per_hour:g} an hour"
        if waited is not None:
            cost += f"; with the wait for the node {waited / 60:.0f} min, at most {waited / 3600 * price_per_hour:.2f}"
        print(cost)

## Trial run

The two small cases. If a part fails, its log is in the folder of the results and in the
output above it.

In [ ]:
trial = submit("trial", SMALL, minutes=40)
trial_waited = wait(trial)

In [ ]:
trial_results = collect(trial)
show(trial_results, trial_waited)

## Full run

All cases. The results are kept in `benchmarks/results/<size of the node>`.

In [ ]:
full = submit("full", ALL, minutes=150)
full_waited = wait(full)

In [ ]:
full_results = collect(full)
show(full_results, full_waited)

kept = ROOT / "benchmarks" / "results" / cluster.size.lower()
shutil.copytree(full_results, kept, dirs_exist_ok=True)
print(f"kept in {kept}")

## Afterwards

A cluster with a minimum of zero nodes releases the node by itself after its idle time. The
environment and the uploaded files stay in the workspace; the results of every job stay in
its output.